In [ ]:
#| default_exp dialog

# Dialog

> Multi-user dialog management, archiving, and AI message routing via dialoghelper.

In [ ]:
#| hide
from nbdev.showdoc import *

每个用户对应一个独立的 solveit dialog（路径 `/wecom/users/{user_id}/chat`），实现多用户隔离。历史会话可归档保留，`/new` 指令触发归档并开启新会话。

In [ ]:
#| export
import re, shutil
from datetime import datetime
from pathlib import Path
from dialoghelper import call_endpa, add_msg
from solveit_wxbot.api import send_text

_DATAPATH = Path('/app/data')

## AI 摘要生成

归档前向 AI 请求一个不超过10字的主题摘要，用于归档文件命名，方便日后检索。失败时回退为 `untitled`。

In [ ]:
#| export
async def _get_summary(
    dlg_path: str  # Solveit dialog path (e.g. '/wecom/users/uid/chat')
) -> str:
    "Ask the AI to summarise the dialog in ≤10 chars; returns a sanitised string or 'untitled'."
    try:
        msg = await add_msg(
            '用10字以内总结这个对话的主题，只输出主题文字，不要任何标点或解释',
            msg_type='prompt', run=True, wait=True, dname=dlg_path)
        summary = re.sub(r'[^\w\u4e00-\u9fff]', '', msg.strip())[:20]
        return summary or 'untitled'
    except Exception:
        return 'untitled'

## 会话归档

`archive_dialog` 生成摘要、停止 dialog 内核，并将 `.ipynb` 文件移至 `archive/` 子目录，文件名格式为 `YYYYMMDD_HHMM_摘要.ipynb`。

In [ ]:
#| export
async def archive_dialog(
    user_id: str  # WeCom user ID whose active dialog should be archived
) -> bool:
    "Generate an AI summary, stop the dialog, and move its .ipynb to the archive folder."
    dlg_path = f'/wecom/users/{user_id}/chat'
    src = DATAPATH / f'{dlg_path.lstrip("/")}.ipynb'
    if not src.exists(): return False

    summary = await _get_summary(dlg_path)
    await call_endpa('stop_kernel_', dname=dlg_path, name=dlg_path.lstrip('/'), json=True)

    ts = datetime.now().strftime('%Y%m%d_%H%M')
    dest_dir = DATAPATH / f'wecom/users/{user_id}/archive'
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dest_dir / f'{ts}_{summary}.ipynb'))
    return True

## 消息路由

`process_message` 是核心入口：收到 `/new` 时归档旧会话并创建新 dialog；其他消息则追加为 prompt 并将 AI 回复发回给用户。

In [ ]:
#| export
async def process_message(
    user_id: str,  # WeCom sender ID
    content: str   # Plain-text message body
):
    "Route messages: '/new' archives and resets the session; anything else is forwarded to AI."
    dlg_path = f'/wecom/users/{user_id}/chat'
    src = DATAPATH / f'{dlg_path.lstrip("/")}.ipynb'
    try:
        if content.strip() == '/new':
            if src.exists():
                ok = await archive_dialog(user_id)
                await send_text(user_id, '✅ 已归档旧对话，新会话已开启' if ok else '⚠️ 归档失败，请重试')
                if not ok: return
            else:
                await send_text(user_id, '✅ 新会话已开启')
            await call_endpa('create_dialog_', dname=dlg_path, name=dlg_path.lstrip('/'), template=True, json=True)
            return

        await call_endpa('create_dialog_', dname=dlg_path, name=dlg_path.lstrip('/'), template=True, json=True)
        reply = await add_msg(content, msg_type='prompt', run=True, wait=True, dname=dlg_path)
        log.info(f'回复 {user_id}: {reply[:10]}...')
        await send_text(user_id, reply)

    except Exception as e:
        log.info(f'❌ 处理失败: {e}')
        await send_text(user_id, '⚠️ 处理失败，请稍后重试')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()